## Random Forest Feature Engineering

This notebook:
1. Loads flight and weather raw data from `1_downloaddata/raw`
2. Creates a `target` label (`Cancelled`, `Delayed`, `On time`)
3. Removes leakage columns
4. Builds hourly weather features
5. Merges weather to flights by origin/date/departure hour
6. Saves a cleaned modeling dataset

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import holidays


In [2]:
df = pd.read_parquet("../../1_download_data/raw/all_flights_2018-2022_raw.parquet")
df.head()


,FlightDate,Airline,Origin,Dest,Cancelled,Diverted,CRSDepTime,DepTime,DepDelayMinutes,DepDelay,...,WheelsOn,TaxiIn,CRSArrTime,ArrDelay,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,DistanceGroup,DivAirportLandings,year
0,2018-01-23,Endeavor Air Inc.,ABY,ATL,False,False,1202,1157.0,0.0,-5.0,...,1249.0,7.0,1304,-8.0,0.0,-1.0,1300-1359,1,0.0,2018
1,2018-01-24,Endeavor Air Inc.,ABY,ATL,False,False,1202,1157.0,0.0,-5.0,...,1246.0,12.0,1304,-6.0,0.0,-1.0,1300-1359,1,0.0,2018
2,2018-01-25,Endeavor Air Inc.,ABY,ATL,False,False,1202,1153.0,0.0,-9.0,...,1251.0,11.0,1304,-2.0,0.0,-1.0,1300-1359,1,0.0,2018
3,2018-01-26,Endeavor Air Inc.,ABY,ATL,False,False,1202,1150.0,0.0,-12.0,...,1242.0,11.0,1304,-11.0,0.0,-1.0,1300-1359,1,0.0,2018
4,2018-01-27,Endeavor Air Inc.,ABY,ATL,False,False,1400,1355.0,0.0,-5.0,...,1448.0,11.0,1500,-1.0,0.0,-1.0,1500-1559,1,0.0,2018


### Create label and remove leakage features

- `target = Cancelled` if cancelled flight  
- `target = Delayed` if departure delay >= 15 min  
- Otherwise `On time`

Then we remove columns that would leak post-departure outcomes into modeling.

In [3]:
df.columns

Index(['FlightDate', 'Airline', 'Origin', 'Dest', 'Cancelled', 'Diverted',
       'CRSDepTime', 'DepTime', 'DepDelayMinutes', 'DepDelay', 'ArrTime',
       'ArrDelayMinutes', 'AirTime', 'CRSElapsedTime', 'ActualElapsedTime',
       'Distance', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek',
       'Marketing_Airline_Network', 'Operated_or_Branded_Code_Share_Partners',
       'DOT_ID_Marketing_Airline', 'IATA_Code_Marketing_Airline',
       'Flight_Number_Marketing_Airline', 'Operating_Airline',
       'DOT_ID_Operating_Airline', 'IATA_Code_Operating_Airline',
       'Tail_Number', 'Flight_Number_Operating_Airline', 'OriginAirportID',
       'OriginAirportSeqID', 'OriginCityMarketID', 'OriginCityName',
       'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac',
       'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'DestCityName',
       'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'DepDel15',
       'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOu

In [4]:
df["FlightDate"] = pd.to_datetime(df["FlightDate"], errors="coerce")
df["CRSDepTime"] = pd.to_numeric(df["CRSDepTime"], errors="coerce").fillna(0).astype(int)

conditions = [
    df["Cancelled"] == True,
    df["DepDelayMinutes"] >= 15
]
choices = ["Cancelled", "Delayed"]
df["target"] = np.select(conditions, choices, default="On time")

#dropping data leakage columns
df = df.drop(
    columns=[
        "DepTime","DepDelay","DepDelayMinutes","DepDel15","DepartureDelayGroups",
        "ArrTime","ArrDelay","ArrDelayMinutes","ArrDel15","ArrivalDelayGroups",
        "Cancelled","Diverted","DivAirportLandings","TaxiOut","TaxiIn","WheelsOff",
        "WheelsOn","ActualElapsedTime","AirTime", "CRSArrTime", "CRSElapsedTime", "year"
    ],
    errors="ignore"
)

#Dropping duplicate columns
df = df.drop(columns = ["Flight_Number_Marketing_Airline", "OriginAirportSeqID", "DestAirportSeqID", 
                        "IATA_Code_Operating_Airline", "Operating_Airline", "IATA_Code_Marketing_Airline",
                        "OriginStateName", "DestStateName"
                       ])


if "Year" in df.columns:
    df = df[df["Year"] != 2020]

print("Flights after engineering:", df.shape)

Flights after engineering: (24171385, 33)


In [5]:
# holidays
us_holidays = holidays.US()
print("Holidays feature started")

# Is Holiday (0/1)
df["is_holiday"] = df["FlightDate"].dt.date.apply(
    lambda x: 1 if x in us_holidays else 0)

Holidays feature started


In [6]:
df["date"] = df["FlightDate"].dt.date
df['dep_hour'] = df['CRSDepTime'] // 100


In [7]:
df["is_morning_peak"] = df["dep_hour"].between(6, 9).astype(int)

# Evening peak: 17–23
df["is_evening_peak"] = df["dep_hour"].between(17, 23).astype(int)

In [8]:
df.head()

,FlightDate,Airline,Origin,Dest,CRSDepTime,Distance,Year,Quarter,Month,DayofMonth,...,DestWac,DepTimeBlk,ArrTimeBlk,DistanceGroup,target,is_holiday,date,dep_hour,is_morning_peak,is_evening_peak
0,2018-01-23,Endeavor Air Inc.,ABY,ATL,1202,145.0,2018,1,1,23,...,34,1200-1259,1300-1359,1,On time,0,2018-01-23,12,0,0
1,2018-01-24,Endeavor Air Inc.,ABY,ATL,1202,145.0,2018,1,1,24,...,34,1200-1259,1300-1359,1,On time,0,2018-01-24,12,0,0
2,2018-01-25,Endeavor Air Inc.,ABY,ATL,1202,145.0,2018,1,1,25,...,34,1200-1259,1300-1359,1,On time,0,2018-01-25,12,0,0
3,2018-01-26,Endeavor Air Inc.,ABY,ATL,1202,145.0,2018,1,1,26,...,34,1200-1259,1300-1359,1,On time,0,2018-01-26,12,0,0
4,2018-01-27,Endeavor Air Inc.,ABY,ATL,1400,145.0,2018,1,1,27,...,34,1400-1459,1500-1559,1,On time,0,2018-01-27,14,0,0


In [9]:
CLEAN_FILE = Path("../../1_download_data/model_ready/RF_V1.parquet")
# Save
CLEAN_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(CLEAN_FILE, index=False)
print(f"Saved cleaned dataset to {CLEAN_FILE}")

Saved cleaned dataset to ../../1_download_data/model_ready/RF_V1.parquet


In [10]:
df.shape

(24171385, 38)